# MVTec AD Data Exploration and Preprocessing

This notebook performs only reproducible MVTec AD inspection, validation, preprocessing, and reporting. Raw data is treated as read-only.

## 1. Environment Setup and Dataset Inspection

Configure paths once, inspect the actual directory structure, and retain a raw-file checksum manifest so later cells can prove the raw dataset was not changed.

In [4]:
from pathlib import Path
from collections import Counter, defaultdict
import hashlib
import numpy as np
from PIL import Image, UnidentifiedImageError

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'datasets').is_dir() else Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / 'datasets/mvtec_ad'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'datasets/mvtec_ad_processed'
IMAGE_SIZE = 256
SUPPORTED_IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}
RGB_RESAMPLE = Image.Resampling.LANCZOS
MASK_RESAMPLE = Image.Resampling.NEAREST


def is_supported_image(path):
    return path.is_file() and path.suffix.lower() in SUPPORTED_IMAGE_EXTENSIONS

def image_files(directory):
    return sorted(path for path in directory.rglob('*') if is_supported_image(path))

def category_directories(root=RAW_DATA_DIR):
    required = {'train', 'test', 'ground_truth'}
    return sorted(path for path in root.iterdir() if path.is_dir() and required.issubset({item.name for item in path.iterdir() if item.is_dir()}))

def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def dataset_manifest(root):
    return {str(path.relative_to(root)): (path.stat().st_size, file_sha256(path)) for path in sorted(root.rglob('*')) if path.is_file()}

CATEGORIES = category_directories()
assert CATEGORIES, f'No valid MVTec categories found under {RAW_DATA_DIR}'
RAW_FILE_MANIFEST = dataset_manifest(RAW_DATA_DIR)

print('MVTec AD Dataset')
print('=' * 50)
print(f'Raw dataset: {RAW_DATA_DIR.resolve()}')
print(f'Categories discovered: {len(CATEGORIES)}')
print('Categories:', ', '.join(category.name for category in CATEGORIES))
for category in CATEGORIES:
    print(f'\nCategory: {category.name}')
    for split in ('train', 'test', 'ground_truth'):
        split_dir = category / split
        class_counts = {subdir.name: len(image_files(subdir)) for subdir in sorted(split_dir.iterdir()) if subdir.is_dir()}
        print(f'  {split}/')
        for class_name, count in class_counts.items():
            print(f'    {class_name}: {count} image files')

root_non_directories = [path.name for path in RAW_DATA_DIR.iterdir() if not path.is_dir()]
print(f'\nRaw file manifest entries: {len(RAW_FILE_MANIFEST)}')
print('Unexpected top-level non-directory entries:', root_non_directories or 'none')

FileNotFoundError: [Errno 2] No such file or directory: 'datasets/mvtec_ad'

## 2. Dataset Statistics

Count every split and class discovered in the raw dataset, then measure decodable image properties without making assumptions about categories or anomaly labels.

In [ ]:
def split_records(split):
    records = []
    for category in CATEGORIES:
        split_dir = category / split
        for path in image_files(split_dir):
            relative = path.relative_to(split_dir)
            records.append({'category': category.name, 'split': split, 'label': relative.parts[0], 'path': path})
    return records

def probe_image(path):
    try:
        with Image.open(path) as image:
            image.load()
            return {'readable': True, 'size': image.size, 'mode': image.mode, 'channels': len(image.getbands()), 'format': image.format, 'error': None}
    except (UnidentifiedImageError, OSError, ValueError) as error:
        return {'readable': False, 'size': None, 'mode': None, 'channels': None, 'format': None, 'error': f'{type(error).__name__}: {error}'}

RAW_RECORDS = {split: split_records(split) for split in ('train', 'test', 'ground_truth')}
IMAGE_STATISTICS = {}
for split, records in RAW_RECORDS.items():
    probes = [probe_image(record['path']) for record in records]
    readable = [probe for probe in probes if probe['readable']]
    IMAGE_STATISTICS[split] = {
        'total': len(records), 'readable': len(readable),
        'formats': Counter(probe['format'] for probe in readable),
        'modes': Counter(probe['mode'] for probe in readable),
        'channels': Counter(probe['channels'] for probe in readable),
        'dimensions': Counter(probe['size'] for probe in readable),
    }

print('Dataset Statistics')
print('=' * 50)
print(f'Categories: {len(CATEGORIES)}')
print(f"Total training images: {len(RAW_RECORDS['train'])}")
print(f"Total test images: {len(RAW_RECORDS['test'])}")
print(f"Total ground-truth masks: {len(RAW_RECORDS['ground_truth'])}")
print('\nPer-category counts')
print(f"{'Category':<14} {'Train':>7} {'Test':>7} {'Good test':>10} {'Anomalous':>10} {'Masks':>7}")
for category in CATEGORIES:
    name = category.name
    train_count = sum(record['category'] == name for record in RAW_RECORDS['train'])
    test_records = [record for record in RAW_RECORDS['test'] if record['category'] == name]
    good_count = sum(record['label'] == 'good' for record in test_records)
    mask_count = sum(record['category'] == name for record in RAW_RECORDS['ground_truth'])
    print(f'{name:<14} {train_count:>7} {len(test_records):>7} {good_count:>10} {len(test_records) - good_count:>10} {mask_count:>7}')
print('\nImage properties by split')
for split, stats in IMAGE_STATISTICS.items():
    dimensions = stats['dimensions']
    widths = [size[0] for size in dimensions for _ in range(dimensions[size])]
    heights = [size[1] for size in dimensions for _ in range(dimensions[size])]
    print(f"{split}: readable={stats['readable']}/{stats['total']}, formats={dict(stats['formats'])}, modes={dict(stats['modes'])}, channels={dict(stats['channels'])}")
    print(f"  dimensions: min={(min(widths), min(heights))}, max={(max(widths), max(heights))}, unique={len(dimensions)}")

## 3. Train Image Validation

Validate every training image without modifying raw data.

In [ ]:
def validate_split(split):
    records, failures, unexpected = [], [], []
    for category in CATEGORIES:
        split_dir = category / split
        unexpected.extend(path for path in split_dir.rglob('*') if path.is_file() and not is_supported_image(path))
    for item in RAW_RECORDS[split]:
        result = {**item, **probe_image(item['path'])}
        records.append(result)
        if not result['readable']: failures.append(result)
    return {'records': records, 'failures': failures, 'unexpected': unexpected}

TRAIN_VALIDATION = validate_split('train')
print('Train Image Validation')
print(f"Total={len(TRAIN_VALIDATION['records'])}, readable={sum(x['readable'] for x in TRAIN_VALIDATION['records'])}, corrupted={len(TRAIN_VALIDATION['failures'])}, unexpected={len(TRAIN_VALIDATION['unexpected'])}")
assert not TRAIN_VALIDATION['failures'], 'Raw train files are invalid; none were changed.'

## 4. Test Image Validation

In [ ]:
TEST_VALIDATION = validate_split('test')
print('Test Image Validation')
for category in CATEGORIES:
    counts = Counter(x['label'] for x in TEST_VALIDATION['records'] if x['category'] == category.name)
    print(f'{category.name}: {dict(sorted(counts.items()))}')
print(f"Total={len(TEST_VALIDATION['records'])}, readable={sum(x['readable'] for x in TEST_VALIDATION['records'])}, corrupted={len(TEST_VALIDATION['failures'])}, unexpected={len(TEST_VALIDATION['unexpected'])}")
assert not TEST_VALIDATION['failures'], 'Raw test files are invalid; none were changed.'

## 5. Ground Truth Validation

In [ ]:
MASK_VALIDATION = validate_split('ground_truth')
expected, found, mismatches = set(), set(), []
for test in TEST_VALIDATION['records']:
    if test['label'] != 'good': expected.add((test['category'], test['label'], test['path'].stem + '_mask'))
for mask in MASK_VALIDATION['records']:
    found.add((mask['category'], mask['label'], mask['path'].stem))
    source = RAW_DATA_DIR / mask['category'] / 'test' / mask['label'] / (mask['path'].stem.removesuffix('_mask') + '.png')
    source_probe = probe_image(source)
    if mask['readable'] and source_probe['readable'] and mask['size'] != source_probe['size']: mismatches.append((mask['path'], source))
missing_masks, unexpected_masks = expected - found, found - expected
print('Ground Truth Validation')
print(f"Masks={len(MASK_VALIDATION['records'])}, corrupt={len(MASK_VALIDATION['failures'])}, missing={len(missing_masks)}, unexpected={len(unexpected_masks)}, dimension mismatches={len(mismatches)}")
assert not MASK_VALIDATION['failures'] and not missing_masks and not unexpected_masks and not mismatches, 'Ground-truth validation failed; raw masks remain unchanged.'

## 6. Preprocessing

In [ ]:
def output_path(path): return (PROCESSED_DATA_DIR / path.relative_to(RAW_DATA_DIR)).with_suffix('.png')
def preprocess(path, mask=False):
    destination = output_path(path); destination.parent.mkdir(parents=True, exist_ok=True)
    with Image.open(path) as source:
        source.load(); image = source.copy() if mask else source.convert('RGB')
        image.resize((IMAGE_SIZE, IMAGE_SIZE), MASK_RESAMPLE if mask else RGB_RESAMPLE).save(destination, 'PNG')

for split in ('train', 'test'):
    for item in RAW_RECORDS[split]: preprocess(item['path'])
for item in RAW_RECORDS['ground_truth']: preprocess(item['path'], mask=True)
RAW_DATASET_UNCHANGED = RAW_FILE_MANIFEST == dataset_manifest(RAW_DATA_DIR)
assert RAW_DATASET_UNCHANGED, 'Raw dataset changed unexpectedly.'
print('Processed RGB images with LANCZOS and masks with nearest-neighbor interpolation.')

## 7. Processed Dataset Validation

In [ ]:
def validate_processed(records, mask=False):
    failures = []
    for item in records:
        probe = probe_image(output_path(item['path']))
        if not probe['readable'] or probe['size'] != (IMAGE_SIZE, IMAGE_SIZE) or (not mask and probe['mode'] != 'RGB'): failures.append(item['path'])
    return failures
train_failures = validate_processed(RAW_RECORDS['train'])
test_failures = validate_processed(RAW_RECORDS['test'])
mask_failures = validate_processed(RAW_RECORDS['ground_truth'], mask=True)
PROCESSED_VALIDATION_PASS = not (train_failures or test_failures or mask_failures)
print('Processed Dataset Validation')
print(f"Train failures={len(train_failures)}, test failures={len(test_failures)}, mask failures={len(mask_failures)}")
assert PROCESSED_VALIDATION_PASS, 'Processed dataset validation failed.'

## 8. Final Dataset Report

In [ ]:
print('MVTec AD Preprocessing Report')
print(f'Categories: {len(CATEGORIES)}')
print(f"Raw train/test/masks: {len(RAW_RECORDS['train'])}/{len(RAW_RECORDS['test'])}/{len(RAW_RECORDS['ground_truth'])}")
print(f'Target image size: {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f"Raw data preserved: {'PASS' if RAW_DATASET_UNCHANGED else 'FAIL'}")
print(f"Ground-truth validation: {'PASS' if not (missing_masks or unexpected_masks or mismatches) else 'FAIL'}")
print(f"Processed dataset validation: {'PASS' if PROCESSED_VALIDATION_PASS else 'FAIL'}")